# 이전 대화를 요청에 넣고 필요한 범위 고르기

이전 대화를 넣었을 때와 넣지 않았을 때의 답변을 비교하고, 모델에게 전달할 최근 메시지를 골라 봅니다. 선택한 기록과 보관한 원본이 어떻게 다른지도 확인합니다.


### 준비. 환경과 메시지 도구

- **메시지 도구**: 역할이 있는 메시지 객체, 대화 목록을 넣는 템플릿, 최근 범위를 고르는 함수 사용.


In [1]:
from pathlib import Path
from dotenv import load_dotenv

DATA_DIR = Path.cwd()
DATA_DIR

WindowsPath('c:/sesac_ddm/lecture/12_ai-agent/06_chatbot_memory')

## 1. 이전 대화를 요청에 넣기

### 1-1. 이력 자리가 있는 요청 틀

- **`MessagesPlaceholder("history")`**: 여러 메시지를 역할과 순서를 유지한 채 채팅 템플릿에 넣는 자리.
- **요청 순서**: 공통 system 지시 → 이전 메시지 목록 → 현재 사용자 질문.
- **`prompt | model`**: 완성한 요청을 모델에 전달하는 체인. 마지막 결과는 `AIMessage` 객체.


In [16]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, trim_messages

model = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,
    max_completion_tokens=300,
    max_retries=0
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "친절한 한국어 도우미"),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])
chain = prompt | model

### 1-2. 첫 자기소개 보내기

- **빈 이력 `[]`**: 전달할 이전 메시지가 없는 첫 요청.
- **반환값 보관**: 실제 첫 답변을 `first_reply`에 받아 다음 이력 구성에 재사용.


In [3]:
introduction = "내 이름은 은지야. 반갑다고 인사해줘"
first_reply = chain.invoke({"history": [], "question": introduction})

### 1-3. 첫 답변의 타입과 본문

- **`AIMessage`**: 모델 답변의 역할·본문 등을 담은 메시지 객체.
- **`.content`**: 메시지 객체에서 읽는 답변 본문. 전체 상태 딕셔너리와 구분.


In [4]:
print(type(first_reply))
print(first_reply.content)

<class 'langchain_core.messages.ai.AIMessage'>
안녕하세요, 은지님! 만나서 정말 반가워요. 어떻게 도와드릴까요?


### 1-4. 이력 없이 이름 묻고 답변 확인

- **이력 없는 요청**: 같은 체인에 질문해도 `history=[]`이면 자기소개를 전달하지 않음.
- **판단 근거**: 답변 문구와 실제 요청에 포함한 정보를 구분. 이름을 우연히 맞혀도 기억의 증거가 되지는 않음.


In [5]:
question = "내 이름이 뭐야?"
without_history = chain.invoke({"history":[], "question": question})

print(without_history.content)

죄송하지만 사용자의 이름을 알 수 없어요. 이름을 알려주시면 기억할게요!


### 1-5. 이전 대화 한 쌍과 완성된 요청

- **`history`**: 사용자 자기소개와 실제 AI 답변을 순서대로 담은 메시지 객체 리스트.
- **`inputs`**: 이전 목록과 현재 질문을 각각의 키에 담은 딕셔너리.
- **템플릿 실행**: `prompt.invoke()` → `ChatPromptValue`, `to_messages()` → 메시지 객체 리스트.
- **현재 질문의 위치**: history 밖의 마지막 사용자 메시지로 한 번만 포함.


In [11]:
history = [HumanMessage(content=introduction), first_reply]
inputs = {"history": history, "question": question}
request_messages = prompt.invoke(inputs).to_messages()

for message in request_messages:
    print(message.type, message.content)

system 친절한 한국어 도우미
human 내 이름은 은지야. 반갑다고 인사해줘
ai 안녕하세요, 은지님! 만나서 정말 반가워요. 어떻게 도와드릴까요?
human 내 이름이 뭐야?


### 1-6. 이력을 넣은 답변과 원본 비교

- **이력을 넣은 요청**: 같은 현재 질문에 자기소개·첫 답변을 함께 전달.
- **체인 실행**: 템플릿 다음 모델까지 실행하여 새 답변 메시지를 받음.
- **원본 리스트**: 새 답변을 받았다고 직접 만든 `history`에 자동 추가되지는 않음.


In [13]:
with_history = chain.invoke(inputs)
print("이력 없음:", without_history.content)
print("이력 있음:", with_history.content)

이력 없음: 죄송하지만 사용자의 이름을 알 수 없어요. 이름을 알려주시면 기억할게요!
이력 있음: 은지님, 당신의 이름은 은지입니다! 기억하고 있어요.


## 2. 최근 메시지 선택과 원본 보존

### 2-1. 선택 범위를 비교할 고정 원본

- **원본 `HISTORY`**: 사용자·AI 메시지 세 쌍, 총 6개의 메시지 객체 리스트.
- **준비된 AI 답변**: 선택 범위를 같은 조건에서 비교하기 위한 예시 문장.
- **현재 질문 `QUESTION`**: 원본 이력과 별도로 둔 ‘그 둘의 차이’ 질문.
- **비교 기준**: 역할·본문을 튜플 목록에 담아 선택 후 원본 내용과 순서가 같은지 확인.


In [15]:
HISTORY = [
    HumanMessage(content="내 이름은 민수야."),
    AIMessage(content="민수님, 반갑습니다."),
    HumanMessage(content="Document Loader가 뭐야?"),
    AIMessage(content="원본에서 텍스트를 읽어 Document로 준비하는 도구입니다."),
    HumanMessage(content="Text Splitter는 뭐야?"),
    AIMessage(content="긴 Document를 검색할 작은 조각으로 나누는 도구입니다."),
]
QUESTION = "그 둘의 차이는 뭐야?"


### 2-2. 최근 메시지 선택과 원본 확인

- **`trim_messages()`**: 원본에서 조건에 맞는 메시지 목록을 선택해 반환하는 함수.
- **개수 기준**: `token_counter=len`이므로 `max_tokens=count`는 실제 토큰 수가 아닌 메시지 개수 기준.
- **선택 규칙**: `last`는 최근 범위, `start_on="human"`은 사용자부터 시작, `allow_partial=False`는 메시지 본문을 부분 절단하지 않음.
- **원본 보존**: 선택은 새로운 목록을 얻는 작업. 원본 메시지의 삭제와 구분.


In [17]:
count = 2

recent = trim_messages(
    HISTORY,
    max_tokens=count,
    token_counter=len,
    strategy="last",
    start_on="human",
    allow_partial=False
)

for message in recent:
    print(f"{message.type}: {message.content}")

print("현재 질문:", QUESTION)
print("선택된 메세지 수:", len(recent))
print("원본 메세지 수 : ", len(HISTORY))

human: Text Splitter는 뭐야?
ai: 긴 Document를 검색할 작은 조각으로 나누는 도구입니다.
현재 질문: 그 둘의 차이는 뭐야?
선택된 메세지 수: 2
원본 메세지 수 :  6
